In [3]:
import pandas as pd
import numpy as np
import altair as alt
from ecostyles import EcoStyles

styles = EcoStyles(); styles.register_and_enable_theme()

yrs = list(range(2030, 2076))
# baseline debt path to ~300%
baseline = np.interp(yrs, [2030,2035,2045,2055,2065,2075], [95,98,117,154,215,300])

# Illustrative additive split of the debt path into three drivers.
# Shares grow over time; at 2075 baseline=300, split e.g.:
#   fixed/structural core ~ 200, demographic ~ 70, changeable policy ~ 30
# (ramp each from a 2030 base where debt=95 is essentially all 'core')
f = np.linspace(0, 1, len(yrs))  # 0 at 2030 -> 1 at 2075
core   = 95 + (205-95)*f*(baseline/ np.interp(yrs,[2030,2075],[95,300]))*0  # keep simple below
# simpler: define endpoint shares and interpolate each band's height
def band(end_2030, end_2075):
    return np.interp(yrs, [2030,2075], [end_2030, end_2075])
core_h   = band(95, 200)   # structural/fixed
demo_h   = band(0, 70)     # demographic-driven
policy_h = baseline - core_h - demo_h   # changeable policy = residual to hit baseline

df = pd.DataFrame({"year":yrs, "Fixed / structural":core_h,
                   "Demographic":demo_h, "Changeable policy":policy_h})
long = df.melt("year", var_name="driver", value_name="pp")
long["date"] = pd.to_datetime(long["year"], format="%Y")
order = ["Fixed / structural","Demographic","Changeable policy"]

area = alt.Chart(long).mark_area().encode(
    x=alt.X("date:T", axis=alt.Axis(format="%Y", tickCount=7), title=None),
    y=alt.Y("pp:Q", title="Public sector net debt (% of GDP)", stack="zero"),
    color=alt.Color("driver:N", sort=order,
        scale=alt.Scale(domain=order, range=["#94a3b8","#36B7B4","#E6224B"]),
        legend=alt.Legend(orient="top", title=None)),
    order=alt.Order("driver:N"))

caption = alt.Title(
    text="Source: author's illustration using OBR, July 2026 Fiscal Risks and Sustainability report",
    subtitle=[
        "Projected UK debt to 2075-76, % of GDP, split by driver (illustrative).",
        "Demographic pressure and changeable policy choices stack on top of the structural path.",
    ],
    orient="bottom", anchor="start", fontSize=11, subtitleFontSize=10,
    color="#676A86", subtitleColor="#676A86", dy=12)

chart = area.properties(width=700, height=360, title=caption).configure(
    background="white", font="Circular Std").configure_view(
    fill="transparent", stroke="transparent")

styles.save(chart, path=".", name="debt_driver_stack", svg=True)
chart.save("debt_driver_stack.png", scale_factor=2.0)
chart

alt.Chart(...)